In [4]:
# Q1
def analyze_transactions(transactions):
    seen_ids = []
    customers = {}
    
    for t in transactions:
        c_name = t['customer']
        if c_name not in customers:
            customers[c_name] = {'total': 0, 'cats': {}}
            
    for t in transactions:
        t_id = t['id']
        if t_id in seen_ids:
            continue  
        seen_ids.append(t_id)
        
        if t['status'] == 'success':
            c_name = t['customer']
            amt = t['amount']
            cat = t['category']
            
            customers[c_name]['total'] += amt
            
            if cat not in customers[c_name]['cats']:
                customers[c_name]['cats'][cat] = 0
            customers[c_name]['cats'][cat] += amt

    ans = []
    for c_name in customers:
        data = customers[c_name]
        
        top_cat = None
        max_val = -1
        for cat in data['cats']:
            if data['cats'][cat] > max_val:
                max_val = data['cats'][cat]
                top_cat = cat
                
        ans.append({
            'customer': c_name,
            'total': data['total'],
            'top_category': top_cat
        })

    def sort_logic(item):
        return (-item['total'], item['customer'])
        
    ans.sort(key=sort_logic)
    
    rank = 1
    for row in ans:
        row['rank'] = rank
        rank += 1
        
    return ans

if __name__ == "__main__":
    sample_input = [
        {'id': 1, 'customer': "Ali", 'category': "Food", 'amount': 500, 'status': "success"},
        {'id': 2, 'customer': "Ali", 'category': "Travel", 'amount': 200, 'status': "failed"},
        {'id': 3, 'customer': "Sara", 'category': "Food", 'amount': 700, 'status': "success"}
    ]
    
    result = analyze_transactions(sample_input)
    print("Task 01 Result:")
    for res in result:
        print(res)

Task 01 Result:
{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}


In [5]:
#Q2

def longest_unique_window(text):
    char_dict = {}
    max_length = 0
    start_index = 0
    best_start = 0
    
    for i in range(len(text)):
        char = text[i]
        
        if char in char_dict and char_dict[char] >= start_index:
            start_index = char_dict[char] + 1
            
        char_dict[char] = i
        current_length = i - start_index + 1
        
        if current_length > max_length:
            max_length = current_length
            best_start = start_index
            
    ans_string = text[best_start : best_start + max_length]
    return ans_string, best_start, max_length

if __name__ == "__main__":
    sample_text = "abcabcbb"
    sub_str, start_idx, length = longest_unique_window(sample_text)
    
    print("Task 02 Result:")
    print(f"Substring: '{sub_str}'")
    print(f"Start Index: {start_idx}")
    print(f"Length: {length}")

Task 02 Result:
Substring: 'abc'
Start Index: 0
Length: 3


In [6]:
#Q3
def find_paths(data, target, path=""):
    paths = []
    
    if isinstance(data, dict):
        for k, v in data.items():
            new_path = f"{path}.{k}" if path else str(k)
            paths.extend(find_paths(v, target, new_path))
            
    elif isinstance(data, list):
        for i, v in enumerate(data):
            paths.extend(find_paths(v, target, f"{path}[{i}]"))
            
    elif type(data) == type(target) and data == target:
        paths.append(path)
        
    return paths

if __name__ == "__main__":
    sample_data = {"a": [{"x": 5}, 5], "b": {"y": 5}}
    print(find_paths(sample_data, 5))

['a[0].x', 'a[1]', 'b.y']


In [7]:
#Q4
def plan_execution(tasks, dependencies):
    in_degree = {t: 0 for t in tasks}
    for u, v in dependencies:
        if u not in tasks or v not in tasks:
            raise ValueError("Unknown task in dependencies")
        in_degree[v] += 1
        
    order = []
    while len(order) < len(tasks):
        available = sorted([t for t in tasks if in_degree[t] == 0 and t not in order])
        
        if not available:
            raise ValueError("Cycle detected, execution not possible")
            
        curr = available[0]
        order.append(curr)
        
        for u, v in dependencies:
            if u == curr:
                in_degree[v] -= 1
                
    return order

if __name__ == "__main__":
    tasks = ["build", "test", "deploy"]
    deps = [("build", "test"), ("test", "deploy")]
    print(plan_execution(tasks, deps))

['build', 'test', 'deploy']


In [8]:
#Q5
def evaluate_expression(expression):
    expression = expression.replace(" ", "")
    
    def parse(tokens):
        stack = []
        num = 0
        sign = '+'
        
        while len(tokens) > 0:
            char = tokens.pop(0)
            
            if char.isdigit():
                num = num * 10 + int(char)
                
            if char == '(':
                num = parse(tokens)
                
            if char in "+-*/)" or len(tokens) == 0:
                if sign == '+': stack.append(num)
                elif sign == '-': stack.append(-num)
                elif sign == '*': stack.append(stack.pop() * num)
                elif sign == '/': stack.append(int(stack.pop() / num))
                
                if char == ')': break
                
                sign = char
                num = 0
                
        return sum(stack)

    return parse(list(expression))

if __name__ == "__main__":
    expr = "-(3 + 2) * 4 + 18 / 3"
    if expr.startswith("-"): expr = "0" + expr
    print(evaluate_expression(expr))

-14


In [9]:
#Q6
import uuid

class Account:
    def __init__(self, acc_id, balance=0):
        self.id = acc_id
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        if amount <= 0: raise ValueError("Amount must be positive")
        self.balance += amount
        self.history.append({"id": str(uuid.uuid4()), "action": "deposit", "amount": amount})

    def withdraw(self, amount):
        if amount <= 0: raise ValueError("Amount must be positive")
        if amount > self.balance: raise ValueError("Insufficient funds")
        self.balance -= amount
        self.history.append({"id": str(uuid.uuid4()), "action": "withdraw", "amount": amount})

class Bank:
    def __init__(self):
        self.accounts = {}
        
    def add_account(self, acc):
        self.accounts[acc.id] = acc
        
    def transfer(self, from_id, to_id, amount):
        self.accounts[from_id].withdraw(amount)
        self.accounts[to_id].deposit(amount)

if __name__ == "__main__":
    bank = Bank()
    acc_a, acc_b = Account("A", 1000), Account("B", 200)
    bank.add_account(acc_a)
    bank.add_account(acc_b)
    bank.transfer("A", "B", 350)
    print(f"A: {acc_a.balance}, B: {acc_b.balance}")

A: 650, B: 550


In [11]:
#Q7
class Employee:
    def __init__(self, name):
        self.name = name
    def calculate_pay(self): 
        pass

class SalariedEmployee(Employee):
    def __init__(self, name, salary):
        super().__init__(name)
        self.salary = salary
    def calculate_pay(self): 
        return self.salary

class HourlyEmployee(Employee):
    def __init__(self, name, rate, hours):
        super().__init__(name)
        self.rate, self.hours = rate, hours
    def calculate_pay(self):
        return self.hours * self.rate if self.hours <= 40 else 40 * self.rate + (self.hours - 40) * self.rate * 1.5

class CommissionEmployee(Employee):
    def __init__(self, name, base, sales, rate):
        super().__init__(name)
        self.base, self.sales, self.rate = base, sales, rate
    def calculate_pay(self): 
        return self.base + (self.sales * self.rate)

def generate_payroll_summary(employees):
    return {emp.name: emp.calculate_pay() for emp in employees}

if __name__ == "__main__":
    employees = [HourlyEmployee("Tariq", 20, 45), CommissionEmployee("Sana", 500, 4000, 0.10)]
    print(generate_payroll_summary(employees))

{'Tariq': 950.0, 'Sana': 900.0}


In [12]:
#Q8
import time

def ttl_cache(seconds):
    def decorator(func):
        cache = {}
        stats = {'hits': 0, 'misses': 0}
        
        def wrapper(*args, **kwargs):
            key = str(args) + str(sorted(kwargs.items()))
            now = time.time()
            
            if key in cache and (now - cache[key]['time']) < seconds:
                stats['hits'] += 1
                return cache[key]['value']
                
            stats['misses'] += 1
            result = func(*args, **kwargs)
            cache[key] = {'value': result, 'time': now}
            return result
            
        wrapper.cache_stats = lambda: stats
        return wrapper
    return decorator

@ttl_cache(seconds=2)
def add_numbers(a, b):
    return a + b

if __name__ == "__main__":
    print(add_numbers(2, 3))
    print(add_numbers(2, 3))

5
5


In [13]:
#Q9
def read_errors(path):
    with open(path, 'r') as file:
        for line_num, line in enumerate(file, 1):
            parts = line.strip().split(' | ')
            if len(parts) == 3 and parts[1] == 'ERROR':
                yield line_num, parts[0], parts[2]

def summarize_errors(path):
    summary = {}
    for _, timestamp, _ in read_errors(path):
        date = timestamp.split(' ')[0]
        summary[date] = summary.get(date, 0) + 1
    return summary

if __name__ == "__main__":
    with open("server.log", "w") as f:
        f.write("2026-11-07 10:30:00 | ERROR | Connection lost\n")
    print(summarize_errors("server.log"))

{'2026-11-07': 1}


In [14]:
#Q10
import csv

def reconcile_inventory(inv_path, mov_path, out_inv, out_rej):
    inventory = {}
    
    with open(inv_path, 'r') as f:
        for row in csv.DictReader(f):
            inventory[row['sku']] = {'name': row['name'], 'qty': int(row['quantity'])}

    rejections = []
    with open(mov_path, 'r') as f:
        for row in csv.DictReader(f):
            sku, mtype, units = row['sku'], row['type'], int(row['units'])
            
            if sku not in inventory or units <= 0 or mtype not in ('IN', 'OUT'):
                rejections.append(row)
                continue
                
            if mtype == 'OUT' and inventory[sku]['qty'] < units:
                rejections.append(row)
                continue
                
            if mtype == 'IN': inventory[sku]['qty'] += units
            if mtype == 'OUT': inventory[sku]['qty'] -= units

    with open(out_inv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['sku', 'name', 'quantity'])
        for sku, data in inventory.items():
            writer.writerow([sku, data['name'], data['qty']])

    if rejections:
        with open(out_rej, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=rejections[0].keys())
            writer.writeheader()
            writer.writerows(rejections)

In [15]:
#Q11
import csv

def process_grades(stu_file, scores_file, out_file):
    students = {}
    
    with open(stu_file, 'r') as f:
        for row in csv.DictReader(f):
            students[row['student_id']] = {'name': row['name'], 'scores': {}}

    with open(scores_file, 'r') as f:
        for row in csv.DictReader(f):
            sid, sub, marks = row['student_id'], row['subject'], int(row['marks'])
            if sid in students and 0 <= marks <= 100 and sub not in students[sid]['scores']:
                students[sid]['scores'][sub] = marks

    results = []
    for sid, data in students.items():
        scores = list(data['scores'].values())
        avg = sum(scores) / len(scores) if scores else 0.0
        grade = 'A' if avg >= 85 else 'B' if avg >= 70 else 'C' if avg >= 55 else 'D' if avg >= 40 else 'F'
        results.append({'id': sid, 'name': data['name'], 'avg': avg, 'grade': grade})

    results.sort(key=lambda x: (-x['avg'], x['id']))
    
    with open(out_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'name', 'avg', 'grade', 'rank'])
        for rank, row in enumerate(results, 1):
            writer.writerow([row['id'], row['name'], row['avg'], row['grade'], rank])

In [16]:
#Q12
class ReversibleRange:
    def __init__(self, start, stop, step):
        if step == 0: raise ValueError("Step cannot be zero")
        self.start = start
        self.stop = stop
        self.step = step
        
        self.length = max(0, (stop - start + (step - 1 if step > 0 else step + 1)) // step)

    def __iter__(self):
        curr = self.start
        for _ in range(self.length):
            yield curr
            curr += self.step

    def __reversed__(self):
        curr = self.start + (self.length - 1) * self.step
        for _ in range(self.length):
            yield curr
            curr -= self.step

    def __len__(self):
        return self.length

    def __contains__(self, val):
        if self.length == 0: return False
        if self.step > 0 and not (self.start <= val < self.stop): return False
        if self.step < 0 and not (self.start >= val > self.stop): return False
        return (val - self.start) % self.step == 0

if __name__ == "__main__":
    my_range = ReversibleRange(2, 11, 3)
    print(list(my_range))
    print(list(reversed(my_range)))
    print(5 in my_range)

[2, 5, 8]
[8, 5, 2]
True


In [17]:
#Q13
class Scheduler:
    def __init__(self):
        self.bookings = []
        
    def book(self, room, start, end, b_id):
        if start >= end:
            raise ValueError("Start must be before end")
            
        for b in self.bookings:
            if b['id'] == b_id: raise ValueError("Duplicate ID")
            
        conflicts = []
        for b in self.bookings:
            if b['room'] == room and start < b['end'] and b['start'] < end:
                conflicts.append(b)
                
        if conflicts:
            return conflicts
            
        self.bookings.append({'room': room, 'start': start, 'end': end, 'id': b_id})
        return []

if __name__ == "__main__":
    schedule = Scheduler()
    print(schedule.book("RoomA", "09:00", "10:00", 1))
    print(schedule.book("RoomA", "09:30", "10:30", 2))

[]
[{'room': 'RoomA', 'start': '09:00', 'end': '10:00', 'id': 1}]


In [18]:
#Q14
import json, os

class ContactManager:
    def __init__(self, filename="contacts.json"):
        self.filename = filename
        self.contacts = self.load_data()
        
    def load_data(self):
        if os.path.exists(self.filename):
            try:
                with open(self.filename, 'r') as f: return json.load(f)
            except json.JSONDecodeError: return {}
        return {}

    def save_data(self):
        with open(self.filename, 'w') as f: json.dump(self.contacts, f, indent=4)

    def add(self, c_id, name, phone, email):
        if str(c_id) in self.contacts: return "ID already exists"
        self.contacts[str(c_id)] = {"name": name, "phone": phone, "email": email}
        self.save_data()
        return "Added successfully"

    def search(self, name_query):
        return {k: v for k, v in self.contacts.items() if name_query.lower() in v['name'].lower()}

if __name__ == "__main__":
    manager = ContactManager("test_contacts.json")
    print(manager.add(1, "Ayesha", "123456", "ayesha@test.com"))
    print(manager.search("aye"))

Added successfully
{'1': {'name': 'Ayesha', 'phone': '123456', 'email': 'ayesha@test.com'}}


In [1]:
#Q15
import uuid

class Product:
    def __init__(self, p_id, price, stock):
        self.id = p_id
        self.price = price
        self.stock = stock

class Inventory:
    def __init__(self, catalog):
        self.products = {k: Product(k, v['price'], v['stock']) for k, v in catalog.items()}

class Order:
    def __init__(self, cart, discount, inventory):
        self.id = str(uuid.uuid4())
        self.cart = cart
        self.total = 0
        
        for p_id, qty in cart.items():
            if p_id not in inventory.products:
                raise ValueError("Product not found")
            if qty <= 0 or inventory.products[p_id].stock < qty:
                raise ValueError("Insufficient stock or invalid quantity")
                
            self.total += inventory.products[p_id].price * qty
            
        self.total = self.total - (self.total * discount)
        
        for p_id, qty in cart.items():
            inventory.products[p_id].stock -= qty

if __name__ == "__main__":
    catalog = {"P1": {"price": 100, "stock": 5}}
    inv = Inventory(catalog)
    order = Order({"P1": 2}, 0.10, inv)
    print(f"Total: {order.total}, Remaining Stock: {inv.products['P1'].stock}")

Total: 180.0, Remaining Stock: 3
